---
jupytext:
  text_representation:
    format_name: myst
kernelspec:
  display_name: Python 3
  language: python
  name: python3
---
# Lesson 2 Ã¢â‚¬â€ NumPy & Pandas for Hydrology

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mr-pablinho/drops-course/blob/main/book/01-python-for-hydrology/02-numpy-pandas.ipynb)

**Learning objectives**  
By the end of this lesson you will be able to:
- [ ] Create and manipulate NumPy arrays to perform vectorised hydrological calculations
- [ ] Use Pandas boolean indexing to filter streamflow records by threshold or date range
- [ ] Aggregate daily data into monthly and seasonal summaries with `groupby`
- [ ] Compute rolling-window statistics to approximate baseflow separation

In [ ]:
# Install course dependencies when running on Google Colab
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', 'https://raw.githubusercontent.com/mr-pablinho/drops-course/main/requirements-colab.txt'
    ], check=True)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
CI = os.environ.get('HYDRO_ML_CI', '0') == '1'

if CI:
    df_raw = pd.read_parquet('../../data/samples/usgs_06803495_daily_2000_2015.parquet')
else:
    import dataretrieval.nwis as nwis
    raw, _ = nwis.get_dv(
        sites='06803495',
        parameterCd='00060',
        start='2000-01-01',
        end='2015-12-31',
    )
    df_raw = raw.reset_index()

df = (
    df_raw
    .rename(columns={'00060_Mean': 'discharge_cfs'})
    .assign(datetime=lambda d: pd.to_datetime(d['datetime']))
    .set_index('datetime')
    .sort_index()
    [['discharge_cfs']]
    .pipe(lambda d: d[d['discharge_cfs'] > 0])
    .dropna()
)

print(f'Loaded {len(df):,} daily records from {df.index.min().date()} to {df.index.max().date()}')

## Introduction

Real hydrological analysis lives between two libraries: **NumPy**, which handles fast
numerical operations on arrays, and **Pandas**, which adds labelled indices, time awareness,
and grouped aggregation on top.

In this lesson we return to the Missouri River discharge record from Lesson 1 and
extract progressively richer information from it Ã¢â‚¬â€ flood thresholds, monthly rhythms,
and a simple baseflow proxy Ã¢â‚¬â€ using nothing but these two libraries.

## 1. NumPy Arrays: Vectorised Arithmetic

NumPy's fundamental object is the **ndarray** Ã¢â‚¬â€ a typed, contiguous block of memory
supporting elementwise operations without Python loops. Every arithmetic operator
(`+`, `*`, `**`, `np.log`, Ã¢â‚¬Â¦) broadcasts across the whole array in a single C-level call.

Three operations that appear repeatedly in hydrology:

| Operation | Hydro use | NumPy call |
|---|---|---|
| Log transform | Linearise skewed flood distributions | `np.log(q)` |
| Percentile | Define flood / drought thresholds | `np.percentile(q, 95)` |
| Cumulative sum | Volume-from-flow conversion | `np.cumsum(q)` |

In [ ]:
q = df['discharge_cfs'].to_numpy()   # extract as a bare NumPy array

# Log transform Ã¢â‚¬â€ notice no loop, no list comprehension
log_q = np.log10(q)

# Percentile thresholds commonly used in hydrology
p10  = np.percentile(q, 10)    # chronic low-flow threshold
p50  = np.percentile(q, 50)    # median (Q50)
p90  = np.percentile(q, 90)    # high-flow threshold
p995 = np.percentile(q, 99.5)  # near-flood threshold

print('Discharge percentiles (cfs)')
print(f'  Q10  (low flow)  : {p10:>10,.0f}')
print(f'  Q50  (median)    : {p50:>10,.0f}')
print(f'  Q90  (high flow) : {p90:>10,.0f}')
print(f'  Q99.5 (near-flood): {p995:>10,.0f}')

# Daily total volume in acre-feet (1 cfs = 1.9835 acre-ft/day)
volume_af = q * 1.9835
total_vol_maf = volume_af.sum() / 1e6
print(f'\nTotal volume (2000Ã¢â‚¬â€œ2015): {total_vol_maf:.1f} million acre-feet')

## 2. Pandas Boolean Indexing

A **boolean mask** is an array of `True`/`False` values of the same length as a Series
or DataFrame. Passing it to `.loc[]` returns only the rows where the condition is `True`.

This is the standard way to identify:
- **Flood days** Ã¢â‚¬â€ days above a design threshold or statistical percentile
- **Drought days** Ã¢â‚¬â€ days below the Q10 low-flow threshold
- **Missing data windows** Ã¢â‚¬â€ gaps that may affect downstream analysis

In [ ]:
# Reuse the Q10/Q90/Q99.5 thresholds from above
flood_mask  = df['discharge_cfs'] >= p995
drought_mask = df['discharge_cfs'] <= p10
high_mask   = df['discharge_cfs'] >= p90

print(f'Flood days   (Ã¢â€°Â¥ Q99.5 = {p995:,.0f} cfs) : {flood_mask.sum():>4} days')
print(f'High-flow days (Ã¢â€°Â¥ Q90 = {p90:,.0f} cfs)  : {high_mask.sum():>4} days')
print(f'Low-flow days  (Ã¢â€°Â¤ Q10 = {p10:,.0f} cfs)  : {drought_mask.sum():>4} days')

# Inspect the actual flood days
flood_days = df.loc[flood_mask, 'discharge_cfs'].sort_values(ascending=False)
print(f'\nTop-5 discharge days:')
for dt, val in flood_days.head().items():
    print(f'  {dt.date()}  {val:>12,.0f} cfs')

# Combine conditions: high flow AND summer (JunÃ¢â‚¬â€œAug)
summer_flood = df[
    (df['discharge_cfs'] >= p90) &
    (df.index.month.isin([6, 7, 8]))
]
print(f'\nHigh-flow days in summer (JunÃ¢â‚¬â€œAug): {len(summer_flood)}')

## 3. GroupBy: Monthly and Seasonal Summaries

Streamflow has strong **seasonality** driven by snowmelt, precipitation patterns, and
vegetation cycles. `DataFrame.groupby()` splits the data into groups, applies a function
to each, and combines the results.

Typical grouping keys in hydrology:
- `df.index.month` Ã¢â‚¬â€ calendar month (1Ã¢â‚¬â€œ12)
- `df.index.year` Ã¢â‚¬â€ calendar year
- A custom water-year label (see Lesson 4)

In [ ]:
# Monthly statistics Ã¢â‚¬â€ mean, median, 90th percentile
monthly = (
    df.groupby(df.index.month)['discharge_cfs']
    .agg(mean='mean', median='median', q90=lambda s: s.quantile(0.90))
    .rename_axis('month')
)

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
monthly.index = month_names

print('Monthly discharge statistics (cfs):\n')
print(monthly.to_string(float_format='{:,.0f}'.format))

# Bar chart of monthly means
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly.index, monthly['mean'] / 1_000,
       color='steelblue', alpha=0.75, label='Mean')
ax.plot(monthly.index, monthly['median'] / 1_000,
        'o-', color='navy', lw=1.8, ms=6, label='Median')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Discharge (kcfs)', fontsize=11)
ax.set_title('Missouri River Ã¢â‚¬â€ Monthly Mean & Median Discharge (2000Ã¢â‚¬â€œ2015)', fontsize=11)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
plt.tight_layout()
plt.show()

## 4. Rolling Windows: A Baseflow Proxy

**Baseflow** is the portion of streamflow sustained by groundwater discharge between
storm events. A simple (though approximate) proxy is a **rolling minimum** over a
short window: short enough to track seasonal trends, long enough to filter individual
storm peaks.

The USGS BFI (Baseflow Index) filter uses a 5-day rolling minimum; many practitioners
also use a 30-day rolling mean to visualise low-frequency variation.

`Series.rolling(window).agg()` computes any function over a sliding window
of `window` observations, aligned to the right edge by default.

In [ ]:
# Zoom into 2003Ã¢â‚¬â€œ2006 to see intra-annual detail without the 2011 flood dominating
sub = df['2003-01-01':'2006-12-31'].copy()

sub['roll7']  = sub['discharge_cfs'].rolling(7,  min_periods=4).mean()
sub['roll30'] = sub['discharge_cfs'].rolling(30, min_periods=20).mean()
sub['bflow']  = sub['discharge_cfs'].rolling(5,  min_periods=3).min()  # BFI proxy

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(sub.index, sub['discharge_cfs'] / 1_000,
                alpha=0.15, color='steelblue')
ax.plot(sub.index, sub['discharge_cfs'] / 1_000,
        color='steelblue', lw=0.7, alpha=0.8, label='Daily')
ax.plot(sub.index, sub['roll7'] / 1_000,
        color='royalblue', lw=1.4, label='7-day rolling mean')
ax.plot(sub.index, sub['roll30'] / 1_000,
        color='navy', lw=2.0, label='30-day rolling mean')
ax.plot(sub.index, sub['bflow'] / 1_000,
        color='sienna', lw=1.5, ls='--', label='5-day rolling min (baseflow proxy)')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Discharge (kcfs)', fontsize=11)
ax.set_title('Missouri River 2003Ã¢â‚¬â€œ2006 Ã¢â‚¬â€ Rolling Statistics & Baseflow Proxy', fontsize=11)
ax.legend(fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
plt.tight_layout()
plt.show()

## Key Takeaways

- **NumPy vectorisation** eliminates Python loops: percentile, log-transform, unit
  conversion, and cumulative volume all reduce to single function calls.
- **Boolean masks** are the natural idiom for event detection Ã¢â‚¬â€ flood days, drought
  periods, and data gaps all start as a `True/False` array applied with `.loc[]`.
- **`groupby(df.index.month)`** decomposes an annual time series into its seasonal
  structure; the resulting statistics expose peak-flow months and dry seasons at a glance.
- **Rolling windows** smooth high-frequency noise and approximate slowly varying signals
  like baseflow; `min_periods` prevents `NaN` at the start of the record.

## Exercise

**Try it yourself:**  
Using the full `df` record (2000Ã¢â‚¬â€œ2015), compute the **annual maximum discharge**
for each calendar year. Then plot a bar chart with year on the x-axis and peak
discharge (in kcfs) on the y-axis.  
Add a horizontal dashed line at the long-term median of the annual maxima.

*Hint: use `df.groupby(df.index.year)['discharge_cfs'].max()` to get the annual series,
then `ax.axhline()` for the horizontal line.*

In [ ]:
# Solution Ã¢â‚¬â€ try on your own first!
ann_max = df.groupby(df.index.year)['discharge_cfs'].max()
median_ann = ann_max.median()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(ann_max.index, ann_max / 1_000, color='steelblue', alpha=0.8,
       label='Annual max discharge')
ax.axhline(median_ann / 1_000, color='darkred', ls='--', lw=1.8,
           label=f'Median annual max: {median_ann:,.0f} cfs')

ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Peak discharge (kcfs)', fontsize=11)
ax.set_title('Missouri River at Omaha Ã¢â‚¬â€ Annual Maximum Discharge 2000Ã¢â‚¬â€œ2015', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Annual max discharge (cfs):')
for yr, val in ann_max.items():
    flag = ' Ã¢â€ Â 2011 flood' if yr == 2011 else ''
    print(f'  {yr}: {val:>12,.0f}{flag}')